Las **Lakeflow Declarative Pipelines** se refieren al enfoque moderno de ingeniería de datos en Databricks, implementado principalmente a través de **Delta Live Tables** (DLT), donde te centras en qué tablas quieres construir y cómo se relacionan, en lugar de en el paso a paso operacional.

Es un cambio de paradigma de la programación tradicional imperativa a la declarativa.

## **Declarativo vs. Imperativo:** El Cambio de Paradigma

| Enfoque                 | Imperativo (Tradicional Spark/Python)                                 | Declarativo (Lakerflow / DLT)                               |
|-------------------------|------------------------------------------------------------------------|-------------------------------------------------------------|
| Enfoque                 | En la ejecución: Escribes código para cómo hacer las cosas (ej. read, transform, write y OPTIMIZE manualmente). | En el resultado: Defines la tabla de destino y su lógica de negocio. |
| Flujo de Control        | Manual. Debes gestionar la orquestación (Airflow, Jobs), la gestión de estados y el manejo de errores. | Automático. El motor DLT gestiona la orquestación, el encendido/pagado de clusters, y el stream o batch de la operación. |
| Creación de Tablas      | `spark.write.format("delto").saveAsTable(...)`                         | `CREATE LIVE TABLE` (SQL) o `@dlt.table` (Python).          |

## **Delta Live Tables (DLT):** La Implementación del Lakeflow
DLT es el framework que te permite construir pipelines declarativos. El engineer se enfoca en la lógica de negocio, y DLT se encarga automáticamente de todo lo demás:

**Orquestación y Dependencias Automáticas**

Simplemente defines las tablas de origen y destino, y DLT deduce la secuencia de ejecución.

**Ejemplo:** Si la Tabla B se define como `SELECT * FROM Tabla A`, DLT sabe que la Tabla A debe actualizarse antes que la Tabla B. DLT construye y gestiona el DAG (Grafo Acíclico Dirigido) por ti.

**Inferencia y Gestión del Modo de Ejecución**

- Al definir una tabla, tú especificas si es una vista (`VIEW`), una tabla de streaming (`STREAMING LIVE TABLE`) o una tabla materializada (`LIVE TABLE`).

- El motor DLT se encarga de aplicar Structured Streaming si los datos se definen como un stream (ej. desde Auto Loader) o de ejecutar un batch si la fuente es estática.

**Calidad de Datos (Expectations)**

- DLT te permite integrar la validación de calidad de datos directamente en la definición de la tabla de forma declarativa, utilizando el concepto de Expectations (Expectativas).

- **Comando:** En lugar de escribir un código de validación aparte, declaras reglas como: **_"El ID del cliente nunca debe ser nulo"._**

### **Streaming Tables**

En Lakeflow Declarative Pipelines, una streaming table es básicamente una tabla manejada por Unity Catalog que además sirve como destino de datos en streaming.

- A una streaming table le podés escribir uno o varios streaming flows (ej: Append o AUTO CDC).

- AUTO CDC solo funciona con streaming tables.

Los flows se pueden definir de dos maneras:

- Explícitamente (primero definís el flow y después lo conectás a la tabla).

- Implícitamente (al definir la tabla, ya indicas el flow que la va a alimentar).

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE 
  catalog.bronze.streaming_table
AS SELECT
  *
FROM read_files(
  "/Volumes/dbacademy/ops/labuser/orders",
  format => "json"
);

CREATE OR REFRESH STREAMING TABLE
  catalog.bronze.table
AS SELECT
  *
FROM STREAM
  catalog.bronze.streaming_table;

### **Materialized Views**

En Lakeflow Declarative Pipelines, una materialized view es **_también una tabla manejada por Unity Catalog, pero a diferencia de las streaming tables funciona como destino batch_** (no streaming).

- Una materialized view recibe uno o varios materialized view flows.

- La diferencia clave con las streaming tables es que **_los flows siempre se definen de forma implícita al crear la materialized view (no se pueden separar como en streaming)_**.

- Su motor procesa solo los datos nuevos o modificados, evitando reprocesar todo.

  - **Streaming table:** Destino en tiempo real, soporta flows de streaming (Append, AUTO CDC).

  - **Materialized view:** Destino batch, actualiza solo con cambios, y los flows van siempre incluidos en la definición.

Se usa Incremental Refresh para evitar reconstruir la view entera cada vez que llegan nuevos datos. Esto está disponible solo en entornos Serverless.

### **View**

Construye una tabla virtual sin datos físicos basado en el resultado de una consulta en un pipeline.

- Se registran como un objeto en Unity Catalog.

### **Temporary View**

Las Temporary Views persisten solo durante el pipeline.

- No se registran como un objeto en Unity Catalog

- Se usan como queries intermedias

In [0]:
%sql
-- View
CREATE VIEW
  catalog.silver.my_view
AS SELECT
  *
FROM
  catalog.bronze.table;

-- Temporary View
CREATE TEMPORARY VIEW
  catalog.silver.my_view
AS SELECT
  *
FROM
  catalog.bronze.table;

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW 
  catalog.silver.materialized_view
AS SELECT
  *
FROM 
  catalog.bronze.table;

## **Conceptos**

### **Flow**

En Lakeflow Declarative Pipelines, un flow es la unidad básica de procesamiento de datos. Puede ser streaming o batch y siempre sigue el patrón: lee de una fuente → aplica lógica → escribe en un destino.

- **Flows estándar:** Usa los mismos modos de Structured Streaming (Append, Update, Complete). Hoy solo está disponible Append.

**Flows especiales de Lakeflow:**

- **AUTO CDC (streaming):** maneja automáticamente eventos CDC fuera de orden y soporta SCD Tipo 1 y 2.

- **Materialized View (batch):** Procesa solo datos nuevos o modificados en la fuente, evitando reprocesar todo.

### **Sink**

Un sink es un destino de streaming.

Tipos soportados hoy:

- Delta tables

- Apache Kafka topics

- Azure EventHubs topics

- Un sink puede recibir uno o varios streaming flows (Append).

El sink es adonde van a parar los datos en tiempo real, ya sea a una tabla Delta o a un sistema de mensajería como Kafka o EventHubs.

### **Pipeline**

Un pipeline es la unidad completa de desarrollo y ejecución.

Un pipeline puede incluir:

- Flows (streaming o batch)

- Streaming tables

- Materialized views

- Sinks

- **Para usarla:** Escribís todo en el código de la pipeline y luego la ejecutás.

Mientras corre, Lakeflow:

- Analiza dependencias entre flows, tablas, vistas y sinks.

- Orquesta automáticamente el orden de ejecución.

- Decide qué se puede ejecutar en paralelo para optimizar rendimiento.

### **Beneficios de Lakeflow Declarative Pipelines**

- **Orquestación automática:** No tenés que preocuparte por el orden ni por los fallos. La pipeline organiza sola los pasos, corre en paralelo lo que se pueda y, si algo falla, reintenta primero la tarea de Spark, después el flow completo y, si hace falta, toda la pipeline.

- **Procesamiento declarativo:** En lugar de escribir cientos o miles de líneas en Spark o Structured Streaming, lo resolvés con unas pocas funciones declarativas. Por ejemplo, la API AUTO CDC ya viene preparada para manejar cambios en los datos (CDC) con soporte de SCD Tipo 1 y 2, y además resuelve sola problemas típicos como eventos fuera de orden o conceptos avanzados de streaming.

- **Procesamiento incremental:** Cuando actualizás materialized views, solo procesa los datos nuevos o modificados. Así se evita volver a procesar todo desde cero y no necesitás código extra para manejar incrementos.

![](https://docs.databricks.com/aws/en/assets/images/dlt-core-concepts-6bc9894d3682035cadac19f3980875ae.png)

In [0]:
%python
@dlt.expect("valido_id", "id_cliente IS NOT NULL")
@dlt.table
def clientes_limpios():
    # ... lógica de selección

**Acción:** DLT puede configurar qué hacer cuando falla una expectativa:

- **Alertar:** Solo registrar el error.

- **Cuarentena:** Enviar la fila fallida a una tabla separada para revisión.

- **Detener:** Fallar el pipeline si la violación es crítica.

**Eficiencia y Mantenimiento (DLM Automático)**

DLT es consciente de las operaciones de Data Lifecycle Management (DLM) y las aplica automáticamente:

- **`OPTIMIZE` Automático:** DLT ejecuta automáticamente comandos de optimización (como `OPTIMIZE` y `ZORDER BY`) según la configuración de la tabla para mantener la eficiencia de lectura.

- **Gestión de Clusters:** DLT gestiona el ciclo de vida de los Job Clusters.

### **Ejemplo de Sintaxis Declarativa (Python)**

Observa cómo defines la relación en lugar de las operaciones:

In [0]:
# 1. Tabla de Origen (Declaración)
@dlt.table
def datos_brutos_streaming():
    # DLT sabe que esto es un stream porque usamos readStream
    return spark.readStream.format("cloudFiles").load("/dataraw/")

# 2. Tabla de Limpieza (Declaración con Dependencia y Calidad)
@dlt.expect("pais_valido", "Pais IN ('USA', 'CAN', 'MEX')")
@dlt.table(comment="Tabla de clientes validados.")
def clientes_limpios():
    # La dependencia es implícita: selecciona de la tabla de arriba
    return (
        dlt.read_stream("datos_brutos_streaming")
        .select("id_cliente", "pais", "nombre")
        .where("id_cliente IS NOT NULL")
    )

En este ejemplo, tú declaras la estructura (`@dlt.table`) y las reglas (`@dlt.expect`), y el motor de DLT se encarga del imperativo: orquestar, ejecutar el stream, aplicar la lógica de calidad y optimizar la tabla Delta final.

## **UI**

- **Pipeline Graph:** Es el DAG gráfico que define el orden de ejecución de las tareas.

- **Dry Run:** Es una ejecución de prueba que valida la lógica de lo que vas a correr, pero sin ejecutar realmente las transformaciones ni escribir datos en el storage.

- **Permission Settings:** Se asignan permisos que pueden tener los usuarios / grupos sobre un pipeline: 

  - **Is owner:** Permisos de dueño

  - **Can manage:** Puede manejar permisos en el pipeline

  - **Can view:** Puede ver el pipeline

  - **Can run:** Puede correr el pipeline

- **Schedule:** Programar la ejecución del pipeline, con UI o CRON JOBS, y elegir si enviará un mail de `START`, `SUCCESS` o `FAIL`.

### Pipeline Configuration

A pipeline's configuration is **a map of key value pairs** that can be used to parametrize your code:

- Improve code readability / maintainability

- Reuse common parameters in multiple pipelines files

**Key:** source

**Value:** "/Volumes/some/path"

In [0]:
%sql
CREATE STREAMING TABLE
  bronze
AS SELECT
  *
FROM STREAM read_files(
  "${source}/orders",
  format=>'JSON'
);

### **Streaming Join**

Un Streaming Join es la capacidad de unir (hacer un `JOIN`) dos fuentes de datos donde, al menos, una de ellas fluye en tiempo real.

En el mundo tradicional (Batch), un Join es fácil porque los datos son estáticos: Spark mira la Tabla A, mira la Tabla B y las junta. Pero en Streaming, los datos están llegando constantemente, lo que hace que el "matching" sea mucho más complejo.

![](https://miro.medium.com/v2/resize:fit:1400/1*v40tibW63tuEP8wB3ZxvkQ.png)

#### **Stream-to-Static Join (El más común)**

Es cuando unes un flujo en vivo (Streaming) con una tabla fija (Batch).

- **Ejemplo:** Tienes un flujo de transacciones bancarias en tiempo real y quieres unirlo con una tabla de clientes (estática) para saber el nombre de quién hizo la compra.

Este es un **Incremental JOIN**, lo que significa que la tabla estática (static table) no necesita ser reprocesada, solo los datos entrantes son evaluados.

#### **Stream-to-Stream Join (El más complejo)**

Es cuando unes dos flujos que están llegando en tiempo real al mismo tiempo.

- **Ejemplo:** Un flujo de clics en una web y un flujo de compras. Quieres saber qué clic terminó en una compra.

- **El gran reto:** ¿Qué pasa si el clic llega a las 10:00 AM pero la compra ocurre a las 10:05 AM? Spark tiene que "guardar" el clic en memoria (State) hasta que llegue la compra para poder unirlos.

### **3. El concepto de "State" y "Watermarking"**

Como no puedes guardar datos en memoria para siempre (te quedarías sin RAM), en los Joins de streaming es obligatorio usar:

- **Watermarking:** Para decirle a Spark: "Espera a la otra parte del Join máximo 10 minutos; si no llega, descarta el dato".

- **Restricciones de tiempo:** Debes definir un intervalo en la condición del Join (ej. `compra.time BETWEEN clic.time AND clic.time + INTERVAL 1 HOUR`).

## **Deploying a Pipeline to Production**

![](https://cdn5.dcbstatic.com/files/d/a/databricks_docebosaas_com/1768366800/JNn_rtv569Pn1Zntq9l1Vg/authoring/687/687_full_slide29_1.jpg)

### **Pipeline Scheduling**

#### **Triggered**

- Can be triggered manually or run on a schedule
- Refreshes selected tables using data available at the start, then stops once the update is complete

#### **Continuous**

- Continuously processes new data to keep streaming tables and materialized tables up to date in near real time
- Monitors dependencies and **updates only when source data changes**

### **Pipeline Event Log**

![](https://cdn5.dcbstatic.com/files/d/a/databricks_docebosaas_com/1768406400/e4N8EnHAYQkTqmOjchaa-w/authoring/687/687_full_slide36_1.jpg)

The Event Log is written as a hidden Delta table

- Located in the pipeline default catalog and schema

**Querying the Declarative Pipeline Event Log**

- Specify the table location 
- Define the table name

In [0]:
%sql
SELECT * FROM <catalog>.<schema>.<event_log_table_name>